# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MrNICEForBonusWinner/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [13]:
query = """
SELECT
    COUNT(*) AS total_rows,

    MIN(gsc_clicks) AS min_clicks,
    AVG(gsc_clicks) AS avg_clicks,
    MAX(gsc_clicks) AS max_clicks,

    MIN(gsc_impressions) AS min_impressions,
    AVG(gsc_impressions) AS avg_impressions,
    MAX(gsc_impressions) AS max_impressions,

    MIN(ga4_pageviews) AS min_pageviews,
    AVG(ga4_pageviews) AS avg_pageviews,
    MAX(ga4_pageviews) AS max_pageviews
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
"""

display(con.sql(query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_clicks,avg_clicks,max_clicks,min_impressions,avg_impressions,max_impressions,min_pageviews,avg_pageviews,max_pageviews
0,9841378,0,0.083508,274,0,28.518119,40084,0,0.217637,875


The main performance metrics are not evenly distributed. Most records have relatively low values, while a smaller number of records have much higher values. This suggests a heavy-tailed distribution, where a few pages receive much more search traffic and engagement than the rest.

The main fields do not appear to be evenly distributed. Most rows have relatively small values, while a smaller number of rows have much larger values. This suggests the data has heavy tails, where a few pages receive much higher search activity than the majority.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [14]:
query = """
SELECT
    CASE
        WHEN gsc_impressions > 1000 THEN 'High impressions'
        ELSE 'Low impressions'
    END AS impression_group,
    AVG(gsc_clicks) AS average_clicks,
    AVG(ga4_pageviews) AS average_pageviews
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY impression_group
ORDER BY impression_group
"""

display(con.sql(query).df())


,impression_group,average_clicks,average_pageviews
0,High impressions,5.073857,7.260603
1,Low impressions,0.067045,0.197205


In [15]:
query = """
SELECT
    CASE
        WHEN gsc_clicks > 10 THEN 'High clicks'
        ELSE 'Low clicks'
    END AS click_group,
    AVG(ga4_pageviews) AS average_pageviews
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY click_group
ORDER BY click_group
"""

display(con.sql(query).df())

,click_group,average_pageviews
0,High clicks,21.432719
1,Low clicks,0.207672


Signal 1: Higher impressions are generally associated with higher clicks.

Verdict: CONFIRMED

The observed data shows that pages with higher impression levels tend to receive more clicks compared with pages with lower impressions.


Signal 2: Pages with more clicks usually have more pageviews.

Verdict: CONFIRMED

The measured results show that pages receiving more search clicks generally also receive higher pageviews.


Signal 3: Higher impressions always lead to higher pageviews.

Verdict: MIXED

The data shows a positive relationship between impressions and pageviews, but the relationship is not consistent for every page.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [16]:
query = """
SELECT
    gsc_clicks,
    gsc_impressions,

    CASE
        WHEN gsc_impressions > 1000 THEN 'High impressions'
        ELSE 'Low impressions'
    END AS impression_group

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
"""

display(con.sql(query).df())

,gsc_clicks,gsc_impressions,impression_group
0,0,20,Low impressions
1,0,1,Low impressions
2,1,125,Low impressions
3,0,7,Low impressions
4,0,11,Low impressions
...,...,...,...
9841373,1,154,Low impressions
9841374,0,27,Low impressions
9841375,0,63,Low impressions
9841376,0,381,Low impressions


The data was grouped based on impression levels to check whether pages with higher impressions receive more clicks. The observed difference between high and low impression groups helps evaluate whether the flag assumption is supported by the data.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The analysis shows that search performance signals can help identify pages that may need attention. A content team can use these observations as decision support to prioritize pages with lower visibility or engagement for further review. These signals should guide investigation rather than be treated as guaranteed outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.